In [ ]:
# Install libraries if needed
!pip install transformers datasets accelerate -q

from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset

# 1. Load GPT-2 small (124M parameters)
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# 2. Load a dataset (here: wikitext-2, small and easy)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# 3. Tokenize the dataset
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize, batched=True, remove_columns=["text"])

# 4. Define training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=1,   # small number just to demo
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True
)

# 5. Setup Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"]
)

# 6. Train
trainer.train()

# Save the fine-tuned model
trainer.save_model("./gpt2-finetuned")